# Verify Multi-Agent Supervisor

This notebook demonstrates the **LangGraph Supervisor Pattern**, where a "Manager" Large Language Model intelligently routes a user's instructions to the appropriate specialized sub-agent (either the Document Assistant or the Movie Discovery Assistant).

### Important Concepts for Developers:

1. **The Hand-Off Mechanism**: When the Supervisor dictates that a sub-agent should take over, `langgraph-supervisor` automatically executes a tool call and injects a synthetic message into the state history (e.g., *"I've transferred you to the document assistant."*). The execution is then passed to the chosen sub-agent.

2. **Gemini Role-Constraints & State Modifiers**: Google's Gemini models enforce strict conversational roles (`Human` ➡ `AI` ➡ `Human` ➡ `AI`). Because the supervisor transfers control by leaving an `AIMessage`/`ToolMessage` at the very end of the history, the sub-agents would normally crash or silently fail. To fix this, we've implemented custom Python **State Modifiers** in `src/ai/agents.py` that dynamically inject a dummy `HumanMessage` to satisfy Gemini's API layout constraints.

3. **Debugging Multi-Agent States**: In single-agent graphs, the final answer is always `response['messages'][-1]`. However, in complex multi-agent graphs you should always iterate through the entire message array to debug what the agents are thinking, as intermediate hand-offs or specific tool outputs might be nested in unexpected locations.

In [ ]:
import setup
setup.init()

In [ ]:
from ai.supervisors import get_supervisor

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver 

checkpointer = InMemorySaver()

In [ ]:
supervisor = get_supervisor(model="gemini-3-flash-preview", checkpointer=checkpointer)

In [ ]:
import uuid 
config = {"configurable": {"user_id": "2", "thread_id": uuid.uuid4()}}

response = supervisor.invoke(
    {"messages": 
        [
            {"role": "user", "content": "what are my recent documents?"}
        ]
    },
    config
)

In [ ]:
response['messages'][-1].content